# 00 — EDA on the Kaggle Mining Process Dataset

Real industrial data from a Brazilian iron-ore concentration plant. We:

1. Load the CSV (737K rows, 24 columns).
2. Inspect timestamp coverage and sampling frequency.
3. Look at sensor distributions and obvious anomalies.
4. Identify the temporal alignment between 20-second sensor data and hourly lab measurements.
5. Compute a baseline (predict the global mean of the target).

**Pre-req:** run `bash scripts/download_data.sh` first.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from frothiq.data.loader import (
    SENSOR_COLS, FEED_COLS, TARGET_COLS,
    load_flotation, temporal_split, detect_constant_lab_measurements
)

## 1. Load the dataset

In [ ]:
data = load_flotation()
df = data.df
print(f'Rows: {len(df):,}')
print(f'Time range: {df.timestamp.min()} → {df.timestamp.max()}')
print(f'Span: {(df.timestamp.max() - df.timestamp.min()).days} days')
df.head()

## 2. Sampling frequency check

Sensors should be sampled every 20 seconds. Lab measurements (% Iron Concentrate, % Silica Concentrate) are hourly but forward-filled to the 20-second resolution.

In [ ]:
deltas = df['timestamp'].diff().dt.total_seconds().dropna()
print('Delta-t between rows (seconds):')
print(deltas.describe())
print()
print(f'Most common delta: {deltas.mode().iloc[0]} seconds')

## 3. Distribution of sensor readings

Quick look at the central tendency and spread of each sensor.

In [ ]:
df[SENSOR_COLS + FEED_COLS + TARGET_COLS].describe().T[['mean', 'std', 'min', '50%', 'max']].round(3)

In [ ]:
# Histograms of the most relevant sensors.
key_cols = ['ore_pulp_ph', 'ore_pulp_density', 'amina_flow', 'starch_flow']
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flat, key_cols):
    ax.hist(df[col].dropna(), bins=40, edgecolor='black', alpha=0.8)
    ax.set_title(col)
    ax.set_xlabel(col); ax.set_ylabel('count')
fig.tight_layout()

## 4. Lab measurement freshness

The lab measurements are hourly but forward-filled. Identify which rows are 'fresh' (a new lab reading) vs 'stale' (forward-filled). This is critical for proper training: ideally we train only on 'fresh' rows or weight them more.

In [ ]:
fresh = detect_constant_lab_measurements(df)
print(f'Total rows: {len(fresh):,}')
print(f'Fresh lab readings: {fresh.sum():,} ({100 * fresh.mean():.2f}%)')
print(f'Forward-filled rows: {(~fresh).sum():,} ({100 * (~fresh).mean():.2f}%)')

## 5. Targets over time

How does concentrate quality evolve over the dataset?

In [ ]:
fresh_df = df[fresh].copy()
fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
axes[0].plot(fresh_df['timestamp'], fresh_df['pct_iron_concentrate'], alpha=0.7, lw=0.8)
axes[0].set_title('% Iron Concentrate (lab measurements only)')
axes[0].set_ylabel('%')
axes[1].plot(fresh_df['timestamp'], fresh_df['pct_silica_concentrate'], alpha=0.7, lw=0.8, color='tab:orange')
axes[1].set_title('% Silica Concentrate (lab measurements only)')
axes[1].set_ylabel('%'); axes[1].set_xlabel('time')
fig.tight_layout()

## 6. Naive baseline

Predict the global mean of the target as the prediction. Anything we model later should beat this.

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

train, val, test = temporal_split(df, train_frac=0.7, val_frac=0.15)

for target in TARGET_COLS:
    train_mean = train[target].mean()
    y_test = test[target].values
    y_pred = np.full_like(y_test, train_mean)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f'{target}: train_mean = {train_mean:.3f}, test RMSE = {rmse:.3f}, MAE = {mae:.3f}, R² = {r2:.3f}')

---

## Next: `01_features.ipynb`

Feature engineering pipeline: rolling statistics over 10-min, 1-h, 3-h windows, lag features, calendar features. Persist gold table for downstream notebooks.